# ReplyDesk — Colab demo

**ReplyDesk** is an async email auto-reply assistant with sentiment + product detection, built on LangChain.

Colab can't run 24×7, but it's perfect for a quick demo:

1. Install the package.
2. Run `replay` over a folder of `.eml` fixtures using a deterministic **fake LLM** (no API key needed).
3. Inspect the audit log to see what *would* have been sent.

For real LLM calls, set your `OPENAI_API_KEY` (or other provider key) via Colab secrets (`user_data_manager`), and set `MODE=live` + IMAP/SMTP credentials.

## 1. Install dependencies

About 30 seconds the first time. All deps are pure-Python and ship wheels.

In [ ]:
import os
import pathlib
import subprocess
import sys

# Install deps (skipped if already present).
deps = ['langchain>=0.3', 'langchain-openai>=0.2', 'pydantic>=2.6',
        'pydantic-settings>=2.2', 'aioimaplib>=1.1', 'aiosmtplib>=3.0',
        'aiosqlite>=0.20', 'rapidfuzz>=3.6', 'beautifulsoup4>=4.12', 'PyYAML>=6.0']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + deps)
print('Installed.')

## 2. Clone / upload the project

Either clone the repo, or upload `replydesk/` from your laptop via the Colab file panel.

In [ ]:

# Try cloning first; fall back to manual upload.
if not os.path.exists('/content/replydesk'):
    !git clone https://github.com/your-org/replydesk /content/replydesk 2>/dev/null || echo 'clone failed - upload replydesk/ via the file panel'

os.chdir('/content/replydesk')
os.makedirs('data', exist_ok=True)
print('Working dir:', os.getcwd())
print('Fixtures available:', sorted(pathlib.Path('tests/fixtures').glob('*.eml')))

## 3. Demo: replay fixtures with the fake LLM

`FAKE_LLM=true` makes the analysis + reply chains return deterministic stubs so we can exercise the whole pipeline (parse → analyze → match → rules → draft reply → audit log) without API keys.

In [ ]:
os.environ['FAKE_LLM'] = 'true'
os.environ['MODE'] = 'dry_run'
os.environ['IMAP_USER'] = 'support@example.com'
os.environ['PRODUCTS_FILE'] = 'products.example.yaml'
os.environ['ALLOWED_DOMAINS'] = 'example.com'
os.environ['DB_PATH'] = '/tmp/replydesk_demo.db'

# Remove a previous DB so the demo starts fresh.
import pathlib

pathlib.Path('/tmp/replydesk_demo.db').unlink(missing_ok=True)

from replydesk.config import get_settings
from replydesk.mail_io import FakeMailSource, parse_email_bytes
from replydesk.main import run_once

settings = get_settings()
items = [parse_email_bytes(p.read_bytes(), settings)
         for p in sorted(pathlib.Path('tests/fixtures').glob('*.eml'))]
results = await run_once(settings, source=FakeMailSource(items).__aiter__())

print(f'\nProcessed {len(results)} email(s):')
for r in results:
    print(f'  {r.message_id[:50]:50} -> {r.decision:8} '
          f'intent={r.analysis.intent if r.analysis else "-"} '
          f'products={[m.catalog_id for m in r.matches.matches if m.catalog_id] if r.matches else []}')

## 4. Inspect the audit log

Every email has a row in SQLite with the outcome, the analysis JSON, and the reply JSON.

In [ ]:
import json

import aiosqlite

async with aiosqlite.connect('/tmp/replydesk_demo.db') as db:
    db.row_factory = aiosqlite.Row
    async with db.execute(
        'SELECT message_id, outcome, created_at, analysis_json, reply_json '
        'FROM processed ORDER BY created_at'
    ) as cur:
        rows = [dict(r) async for r in cur.fetchall()]

print(f'Audit log: {len(rows)} rows\n')
for r in rows:
    print(f'  {r["message_id"][:50]:50} outcome={r["outcome"]}')
    if r['analysis_json']:
        a = json.loads(r['analysis_json'])
        print(f'     intent={a["intent"]} sentiment={a["sentiment"]} '
              f'urgency={a["urgency"]} confidence={a["confidence"]:.2f}')
    if r['reply_json']:
        rep = json.loads(r['reply_json'])
        print(f'     reply_subject="{rep["subject"]}" needs_human={rep["needs_human"]}')
    print()

## 5. Going live (optional)

To use a real LLM and a real mailbox, you'll need:

1. **API key** — store via Colab secrets and load via:
   ```python
   from google.colab import userdata
   os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
   ```
2. **IMAP / SMTP** — set `IMAP_HOST`, `IMAP_USER`, `IMAP_PASSWORD`, `SMTP_HOST`, `SMTP_USER`, `SMTP_PASSWORD`.
   Gmail and Outlook require an **app password**, not the account password.
3. **Mode** — start with `MODE=draft_only` and watch the IMAP Drafts folder for a few days before `MODE=live`.
4. **Catalog** — edit `products.yaml` with your real product names, aliases, SKU patterns, and FAQ snippets.

Then call `await run_once()` from this notebook to process the current batch of unseen emails.